In [1]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from thefuzz import process, fuzz

class NeuralMusicEngine:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL MUSIC CORE // V4.0")
        self.root.geometry("1200x850")
        
        # --- PRO TECH PALETTE ---
        self.colors = {
            'void': '#050508',          # Pure Black
            'neon_cyan': '#00f2ff',     # High Energy
            'neon_magenta': '#ff007a',  # Melancholic/Vibe
            'glass_panel': '#11111d',   # Semi-transparent feel
            'text_high': '#ffffff',
            'text_low': '#4a4a6a'
        }
        
        self.music_df = None
        self.current_mood = "CALIBRATING"
        self.setup_ui()
        self.load_engine()

    def setup_ui(self):
        self.root.configure(bg=self.colors['void'])

        # --- LEFT: STATUS & VIBE PANEL ---
        self.side_panel = tk.Frame(self.root, bg=self.colors['glass_panel'], width=300)
        self.side_panel.pack(side=tk.LEFT, fill=tk.Y, padx=5, pady=5)

        tk.Label(self.side_panel, text="NEURAL STATUS", font=('Orbitron', 10, 'bold'), 
                 fg=self.colors['neon_cyan'], bg=self.colors['glass_panel']).pack(pady=(40, 10))

        self.mood_display = tk.Label(self.side_panel, text=self.current_mood, font=('Consolas', 14, 'bold'),
                                     fg=self.colors['text_high'], bg=self.colors['void'], width=15, pady=10)
        self.mood_display.pack(pady=20)

        self.stats_box = tk.Text(self.side_panel, bg=self.colors['void'], fg=self.colors['neon_magenta'],
                                font=('Consolas', 9), height=10, borderwidth=0, padx=10)
        self.stats_box.pack(fill=tk.X, padx=20)
        self.stats_box.insert('1.0', "> System Online\n> Frequency: 44.1kHz\n> Ready for input...")

        # --- RIGHT: MAIN INTERACTIVE STAGE ---
        self.stage = tk.Frame(self.root, bg=self.colors['void'])
        self.stage.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        # Header Image/Text Area
        self.header = tk.Label(self.stage, text="SOUNDSCAPE AI", font=('Orbitron', 32, 'bold'),
                               fg=self.colors['text_high'], bg=self.colors['void'])
        self.header.pack(pady=40)

        # Recommendation Display (The "Human" Response Area)
        self.display_card = tk.Frame(self.stage, bg=self.colors['glass_panel'], pady=30, padx=30)
        self.display_card.pack(fill=tk.BOTH, expand=True, padx=50, pady=10)

        self.resp_label = tk.Label(self.display_card, text="How are we feeling today? Speak your mind...",
                                   font=('Segoe UI Light', 18), fg=self.colors['text_high'],
                                   bg=self.colors['glass_panel'], wraplength=600, justify="center")
        self.resp_label.pack(expand=True)

        # --- INPUT FIELD (Cyber-Bar) ---
        input_container = tk.Frame(self.stage, bg=self.colors['void'], pady=40)
        input_container.pack(fill=tk.X)

        self.entry_frame = tk.Frame(input_container, bg=self.colors['neon_cyan'], padx=1, pady=1)
        self.entry_frame.pack(padx=80, fill=tk.X)

        self.user_input = tk.Entry(self.entry_frame, font=('Segoe UI', 16), 
                                   bg=self.colors['void'], fg=self.colors['neon_cyan'],
                                   insertbackground=self.colors['neon_cyan'], relief=tk.FLAT)
        self.user_input.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=15, pady=10)
        self.user_input.bind("<Return>", lambda e: self.process_interaction())

        # --- ANIMATED FREQUENCY VISUALIZER ---
        self.viz = tk.Canvas(self.root, height=80, bg=self.colors['void'], highlightthickness=0)
        self.viz.pack(side=tk.BOTTOM, fill=tk.X)
        self.animate_viz()

    def animate_viz(self):
        """Creates a professional colorful moving waveform"""
        self.viz.delete("all")
        w = self.root.winfo_width()
        for i in range(0, w, 20):
            h = random.randint(10, 60)
            color = self.colors['neon_cyan'] if h < 40 else self.colors['neon_magenta']
            self.viz.create_rectangle(i, 80-h, i+10, 80, fill=color, outline="")
        self.root.after(100, self.animate_viz)

    def load_engine(self):
        try:
            self.music_df = pd.read_csv('music_sentiment_dataset.csv')
        except:
            messagebox.showerror("Error", "Dataset not found.")

    def process_interaction(self):
        raw_text = self.user_input.get()
        if not raw_text: return
        self.user_input.delete(0, tk.END)
        
        self.resp_label.config(text="Scanning Vibe...", fg=self.colors['text_low'])
        threading.Thread(target=self.generate_human_reply, args=(raw_text,)).start()

    def generate_human_reply(self, text):
        time.sleep(0.8) # Simulate "listening"
        text_lower = text.lower()
        
        # 1. Logic for Matching
        match = self.find_match(text_lower)
        
        # Human-like conversational wrappers
        intros = ["I hear you. This might fit the mood:", "Let's try this energy:", "Based on that, I'm feeling this:"]
        
        if match is not None:
            self.current_mood = match['Mood'].upper()
            self.root.after(0, lambda: self.update_ui_with_match(match, random.choice(intros)))
        else:
            self.root.after(0, lambda: self.resp_label.config(text="I'm not quite catching that frequency. Tell me more about your mood.", fg=self.colors['text_high']))

    def update_ui_with_match(self, match, intro):
        self.mood_display.config(text=self.current_mood, fg=self.colors['neon_cyan'])
        full_text = f"{intro}\n\n'{match['Song_Name']}' by {match['Artist']}\n({match['Genre']} | {match['Tempo (BPM)']} BPM)"
        self.resp_label.config(text=full_text, fg=self.colors['neon_cyan'])
        
        # Update sidebar log
        log_entry = f"> Found: {match['Song_Name']}\n> Tempo: {match['Tempo (BPM)']}BPM\n---\n"
        self.stats_box.insert('1.0', log_entry)

    def find_match(self, query):
        # Checks song, artist, and mood labels
        choices = self.music_df['Song_Name'].tolist() + self.music_df['Mood'].unique().tolist()
        best_match, score = process.extractOne(query, choices, scorer=fuzz.token_set_ratio)
        
        if score > 70:
            # Return the first row that matches either title or mood
            if best_match in self.music_df['Song_Name'].values:
                return self.music_df[self.music_df['Song_Name'] == best_match].iloc[0]
            else:
                return self.music_df[self.music_df['Mood'] == best_match].sample(1).iloc[0]
        return None

if __name__ == "__main__":
    root = tk.Tk()
    app = NeuralMusicEngine(root)
    root.mainloop()

In [2]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from thefuzz import process, fuzz

class EmotionalMusicEngine:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL EMOTION CORE // V5.0")
        self.root.geometry("1200x850")
        
        # Base Tech Palette
        self.colors = {
            'void': '#050508',
            'glass': '#11111d',
            'text': '#ffffff',
            'Happy': '#1DB954',    # Green
            'Sad': '#1e3a8a',      # Deep Blue
            'Relaxed': '#7000ff',  # Purple
            'Motivated': '#ff4500' # Orange/Red
        }
        
        # Sentiment Dictionary to help the "Human" understanding
        self.sentiment_map = {
            'sad': ['miserable', 'unhappy', 'crying', 'heartbroken', 'lonely', 'depressed', 'gloomy', 'down', 'hurt', 'pain'],
            'happy': ['joyful', 'great', 'excellent', 'amazing', 'excited', 'wonderful', 'cheerful', 'content', 'good'],
            'relaxed': ['chill', 'stressed', 'tired', 'calm', 'peaceful', 'sleepy', 'quiet', 'meditating', 'zen'],
            'motivated': ['pumped', 'working out', 'gym', 'focus', 'ready', 'energetic', 'powerful', 'strong', 'grind']
        }

        self.music_df = None
        self.setup_ui()
        self.load_engine()

    def setup_ui(self):
        self.root.configure(bg=self.colors['void'])

        # --- DYNAMIC BACKGROUND CANVAS ---
        self.bg_canvas = tk.Canvas(self.root, bg=self.colors['void'], highlightthickness=0)
        self.bg_canvas.place(x=0, y=0, relwidth=1, relheight=1)

        # --- INTERACTIVE CONTENT LAYER ---
        self.main_frame = tk.Frame(self.root, bg='', bd=0)
        self.main_frame.place(relx=0.5, rely=0.4, anchor='center')

        self.status_label = tk.Label(self.main_frame, text="CORE // OFFLINE", font=('Orbitron', 10), 
                                     fg='#4a4a6a', bg=self.colors['void'])
        self.status_label.pack()

        self.display_card = tk.Frame(self.main_frame, bg=self.colors['glass'], padx=40, pady=40)
        self.display_card.pack(pady=20)

        self.resp_label = tk.Label(self.display_card, 
                                   text="I'm listening. Tell me exactly how you feel right now...",
                                   font=('Segoe UI Light', 22), fg='white',
                                   bg=self.colors['glass'], wraplength=700, justify="center")
        self.resp_label.pack()

        # --- INPUT BAR ---
        self.input_container = tk.Frame(self.root, bg=self.colors['void'], pady=30)
        self.input_container.pack(side=tk.BOTTOM, fill=tk.X)

        self.entry_box = tk.Entry(self.input_container, font=('Segoe UI', 18), 
                                  bg='#1a1a2e', fg='white', insertbackground='white',
                                  relief=tk.FLAT, justify='center')
        self.entry_box.pack(fill=tk.X, padx=150, pady=10)
        self.entry_box.bind("<Return>", lambda e: self.analyze_input())

    def load_engine(self):
        try:
            self.music_df = pd.read_csv('music_sentiment_dataset.csv')
            self.status_label.config(text="CORE // EMOTIONAL ENGINE ONLINE", fg=self.colors['Happy'])
        except:
            messagebox.showerror("Error", "Dataset missing!")

    def update_theme(self, sentiment):
        """Changes the background glow based on emotion"""
        color = self.colors.get(sentiment, self.colors['void'])
        
        # Clear previous glow
        self.bg_canvas.delete("glow")
        
        # Draw soft gradient ovals to create a "Music Vibe" atmosphere
        self.bg_canvas.create_oval(-300, 300, 600, 1000, fill=color, outline="", width=0, tags="glow")
        self.bg_canvas.create_oval(700, -200, 1500, 600, fill=color, outline="", width=0, tags="glow")
        
        self.status_label.config(fg=color)

    def analyze_input(self):
        user_text = self.entry_box.get().lower()
        self.entry_box.delete(0, tk.END)
        
        self.resp_label.config(text="Processing your feelings...", fg="#4a4a6a")
        threading.Thread(target=self.logic_thread, args=(user_text,)).start()

    def logic_thread(self, text):
        time.sleep(1.2) # Human-like reflection pause
        
        detected_sentiment = None
        
        # Advanced keyword mapping to find the hidden emotion
        for main_mood, synonyms in self.sentiment_map.items():
            if any(word in text for word in synonyms) or main_mood in text:
                detected_sentiment = main_mood.capitalize()
                break
        
        # If no keywords found, use fuzzy matching on the Sentiment Label column
        if not detected_sentiment:
            labels = self.music_df['Sentiment_Label'].unique().tolist()
            match_label, score = process.extractOne(text, labels, scorer=fuzz.token_set_ratio)
            if score > 60:
                detected_sentiment = match_label
            else:
                detected_sentiment = "Relaxed" # Default calming vibe

        # Filter the CSV for the real songs matching that feeling
        subset = self.music_df[self.music_df['Sentiment_Label'] == detected_sentiment]
        
        if not subset.empty:
            match = subset.sample(1).iloc[0]
            self.root.after(0, lambda: self.apply_result(match, detected_sentiment))
        else:
            self.root.after(0, lambda: self.resp_label.config(text="I'm here for you. Can you describe that feeling a bit more?", fg="white"))

    def apply_result(self, match, sentiment):
        self.update_theme(sentiment)
        
        # Empathetic, human responses
        dialogue = {
            'Sad': "I'm sorry you're feeling down. Sometimes the right song can hold space for us. Try this...",
            'Happy': "I love that energy! Let's celebrate that mood with this track:",
            'Motivated': "You've got that fire! This should help you keep pushing:",
            'Relaxed': "Let's find some peace. Take a breath and listen to this..."
        }
        
        # Construct the "Human" response
        response = f"{dialogue.get(sentiment, 'Here is something for you:')}\n\n"
        response += f"🎵 {match['Song_Name']}\n"
        response += f"👤 {match['Artist']}\n\n"
        response += f"This is a {match['Mood'].lower()} {match['Genre']} track to help you feel better."
        
        self.resp_label.config(text=response, fg='white')

if __name__ == "__main__":
    root = tk.Tk()
    app = EmotionalMusicEngine(root)
    root.mainloop()

In [4]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from thefuzz import process, fuzz

class MuseIntelligence:
    def __init__(self, root):
        self.root = root
        self.root.title("MUSE AI // EMOTIONAL NEURAL INTERFACE")
        self.root.geometry("1200x850")
        
        # Professional Cinematic Palette
        self.colors = {
            'void': '#020205',
            'panel': '#0a0a0f',
            'Happy': '#00ff88',    # Neo-Mint
            'Sad': '#0066ff',      # Electric Cobalt
            'Relaxed': '#bd00ff',  # Deep Violet
            'Motivated': '#ff3300' # Solar Flare
        }
        
        # Deep Sentiment Mapping (Human Feeling Training)
        self.feeling_training = {
            'Sad': ['lonely', 'miserable', 'heartbroken', 'depressed', 'gloomy', 'crying', 'failed', 'empty', 'lost', 'tired of everything', 'hurting'],
            'Happy': ['amazing', 'wonderful', 'joy', 'blessed', 'winning', 'excited', 'party', 'love', 'grateful', 'smiling', 'ecstatic'],
            'Relaxed': ['zen', 'chill', 'peaceful', 'stormy night', 'quiet', 'meditating', 'soft', 'calm', 'drifting', 'sleepy'],
            'Motivated': ['beast mode', 'gym', 'hustle', 'success', 'power', 'strong', 'grind', 'focus', 'unstoppable', 'ambition']
        }

        self.music_df = None
        self.current_sentiment = "Relaxed"
        self.setup_ui()
        self.load_engine()

    def setup_ui(self):
        self.root.configure(bg=self.colors['void'])

        # --- ATMOSPHERIC CANVAS (The "Music Background") ---
        self.bg_viz = tk.Canvas(self.root, bg=self.colors['void'], highlightthickness=0)
        self.bg_viz.place(x=0, y=0, relwidth=1, relheight=1)

        # --- CENTRAL INTELLIGENCE HUB ---
        self.glass_card = tk.Frame(self.root, bg=self.colors['panel'], padx=50, pady=50)
        self.glass_card.place(relx=0.5, rely=0.45, anchor='center')

        # Visual Representation of Feeling
        self.mood_art = tk.Canvas(self.glass_card, width=300, height=180, bg=self.colors['panel'], highlightthickness=0)
        self.mood_art.pack(pady=(0, 20))

        self.ai_speech = tk.Label(self.glass_card, 
                                   text="I am Muse. Tell me what is weighing on your heart or lifting your spirit...",
                                   font=('Helvetica Neue Light', 20), fg='white',
                                   bg=self.colors['panel'], wraplength=650, justify="center")
        self.ai_speech.pack()

        # --- COMMAND INPUT ---
        self.input_container = tk.Frame(self.root, bg=self.colors['void'], pady=40)
        self.input_container.pack(side=tk.BOTTOM, fill=tk.X)

        self.user_entry = tk.Entry(self.input_container, font=('Helvetica Neue', 18), 
                                   bg='#111116', fg='white', insertbackground='white',
                                   relief=tk.FLAT, justify='center', borderwidth=10)
        self.user_entry.pack(fill=tk.X, padx=200)
        self.user_entry.bind("<Return>", lambda e: self.interact())

        self.animate_environment()

    def load_engine(self):
        try:
            self.music_df = pd.read_csv('music_sentiment_dataset.csv')
        except:
            messagebox.showerror("Error", "Core Database Missing.")

    def animate_environment(self):
        """Creates a professional animated musical background"""
        self.bg_viz.delete("particle")
        w, h = self.root.winfo_width(), self.root.winfo_height()
        color = self.colors.get(self.current_sentiment, '#ffffff')
        
        # Generate 'Musical Particles'
        for _ in range(15):
            x = random.randint(0, w)
            y = random.randint(0, h)
            size = random.randint(2, 5)
            self.bg_viz.create_oval(x, y, x+size, y+size, fill=color, outline="", tags="particle")
            
        self.root.after(100, self.animate_environment)

    def draw_mood_picture(self, sentiment):
        """Generates a professional abstract visual representing the feeling"""
        self.mood_art.delete("all")
        c = self.colors[sentiment]
        
        if sentiment == 'Sad':
            # Vertical 'Falling' Lines (Rain/Tears)
            for i in range(0, 300, 15):
                length = random.randint(20, 100)
                self.mood_art.create_line(i, 0, i, length, fill=c, width=2)
        elif sentiment == 'Happy':
            # Expanding Radiance (Sun/Burst)
            self.mood_art.create_oval(100, 40, 200, 140, outline=c, width=3)
            for i in range(0, 360, 30):
                self.mood_art.create_line(150, 90, 150, 10, fill=c, width=2, tags="ray")
        elif sentiment == 'Motivated':
            # Sharp Kinetic Vectors (Energy/Speed)
            for _ in range(10):
                x1, y1 = random.randint(0, 200), random.randint(0, 150)
                self.mood_art.create_line(x1, y1, x1+100, y1-50, fill=c, width=4, arrow=tk.LAST)
        else: # Relaxed
            # Soft Horizontal Waves
            self.mood_art.create_arc(50, 50, 250, 150, outline=c, style=tk.ARC, width=3)

    def interact(self):
        query = self.user_entry.get()
        if not query: return
        self.user_entry.delete(0, tk.END)
        
        self.ai_speech.config(text="Processing frequency...", fg="#444")
        threading.Thread(target=self.think, args=(query,)).start()

    def think(self, text):
        time.sleep(1.2) # Gemini-style 'thought' delay
        
        detected = "Relaxed"
        for mood, words in self.feeling_training.items():
            if any(w in text.lower() for w in words):
                detected = mood
                break
        
        self.current_sentiment = detected
        subset = self.music_df[self.music_df['Sentiment_Label'] == detected]
        song = subset.sample(1).iloc[0] if not subset.empty else self.music_df.iloc[0]

        # Professional Conversational Responses
        responses = {
            'Sad': f"I hear the heaviness in your words. It's okay to not be okay right now. Music has a way of sitting with us in the dark. I've chosen '{song['Song_Name']}' by {song['Artist']} for you. Let the melody carry some of that weight.",
            'Happy': f"That energy is incredible! It's like the world is in full color for you right now. To celebrate this moment, you have to hear '{song['Song_Name']}' by {song['Artist']}. It perfectly matches your light!",
            'Motivated': f"I can feel your focus—it's razor sharp. You're in the zone to conquer whatever is next. I'm locking in '{song['Song_Name']}' by {song['Artist']} to fuel that fire. Go get it.",
            'Relaxed': f"Taking a moment to breathe is the most productive thing you can do right now. Let's slow down the clock. I recommend '{song['Song_Name']}' by {song['Artist']} to help you find your center."
        }

        self.root.after(0, lambda: self.update_ui(responses[detected], detected))

    def update_ui(self, msg, sentiment):
        self.ai_speech.config(text=msg, fg='white')
        self.draw_mood_picture(sentiment)

if __name__ == "__main__":
    root = tk.Tk()
    app = MuseIntelligence(root)
    root.mainloop()

In [5]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from PIL import Image, ImageTk, ImageDraw, ImageFilter
from thefuzz import process, fuzz

class OmniEmotionPro:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL OMNI-CORE V8.0 // PROFESSIONAL MUSIC AI")
        self.root.geometry("1200x850")
        
        # UI Colors
        self.colors = {
            'void': '#020205',
            'glass': '#0a0a0f',
            'Happy': '#1DB954',
            'Sad': '#3b82f6',
            'Relaxed': '#8b5cf6',
            'Motivated': '#f97316'
        }

        # Human Sentiment Mapping (Training)
        self.lexicon = {
            'Sad': ['lonely', 'miserable', 'heartbroken', 'gloomy', 'crying', 'failed', 'empty', 'lost', 'hurt', 'pain', 'burnout', 'nostalgic'],
            'Happy': ['joyful', 'great', 'amazing', 'blessed', 'winning', 'excited', 'party', 'love', 'grateful', 'smiling', 'ecstatic', 'hype'],
            'Relaxed': ['zen', 'chill', 'peaceful', 'calm', 'drifting', 'sleepy', 'quiet', 'meditating', 'stressed', 'anxious', 'tired'],
            'Motivated': ['beast mode', 'gym', 'hustle', 'success', 'power', 'strong', 'grind', 'focus', 'unstoppable', 'ambition', 'ready']
        }

        self.music_df = None
        self.current_vibe = "Relaxed"
        
        # 1. Prepare Assets (Generates professional images if missing)
        self.prepare_assets()
        
        # 2. Build UI
        self.setup_ui()
        
        # 3. Load Dataset
        self.load_data()

    def prepare_assets(self):
        """Generates professional background and mood images internally."""
        try:
            # Create professional background
            bg = Image.new('RGB', (1200, 850), (5, 5, 15))
            draw = ImageDraw.Draw(bg)
            draw.rectangle([0, 0, 1200, 425], fill=(10, 5, 25))
            bg = bg.filter(ImageFilter.GaussianBlur(radius=30))
            bg.save('bg_music.png')

            # Create mood glows
            for mood, color in [('Happy', (29, 185, 84)), ('Sad', (59, 130, 246)), 
                                ('Relaxed', (139, 92, 246)), ('Motivated', (249, 115, 22))]:
                img = Image.new('RGBA', (400, 400), (0,0,0,0))
                d = ImageDraw.Draw(img)
                # Soft radial glow
                for i in range(200, 0, -1):
                    alpha = int(180 * (1 - (i/200)))
                    d.ellipse([200-i, 200-i, 200+i, 200+i], fill=color + (alpha,))
                img.save(f'{mood.lower()}.png')
        except Exception as e:
            print(f"Asset Gen Warning: {e}")

    def setup_ui(self):
        # Load Background Image
        self.bg_image = ImageTk.PhotoImage(Image.open('bg_music.png'))
        self.bg_label = tk.Label(self.root, image=self.bg_image)
        self.bg_label.place(x=0, y=0, relwidth=1, relheight=1)

        # Central Card (Glassmorphism)
        self.card = tk.Frame(self.root, bg=self.colors['glass'], padx=40, pady=40)
        self.card.place(relx=0.5, rely=0.45, anchor='center')

        # Mood Picture (Real Image Area)
        self.mood_label = tk.Label(self.card, bg=self.colors['glass'])
        self.mood_label.pack(pady=(0, 20))
        self.update_mood_image("relaxed")

        # Conversational Display
        self.ai_text = tk.Label(self.card, 
                                 text="I am your Neural Muse. Tell me everything you're feeling. I am listening.",
                                 font=('Helvetica Neue Light', 20), fg='white',
                                 bg=self.colors['glass'], wraplength=600, justify="center")
        self.ai_text.pack()

        # Input Interface
        input_frame = tk.Frame(self.root, bg=self.colors['void'], pady=30)
        input_frame.pack(side=tk.BOTTOM, fill=tk.X)
        
        self.entry = tk.Entry(input_frame, font=('Helvetica Neue', 18), 
                              bg='#111827', fg='white', insertbackground='white',
                              relief=tk.FLAT, justify='center')
        self.entry.pack(fill=tk.X, padx=150, pady=10)
        self.entry.bind("<Return>", lambda e: self.process())

    def update_mood_image(self, mood_name):
        path = f"{mood_name.lower()}.png"
        img = Image.open(path).resize((250, 250), Image.LANCZOS)
        self.current_img = ImageTk.PhotoImage(img)
        self.mood_label.config(image=self.current_img)

    def load_data(self):
        try:
            self.music_df = pd.read_csv('music_sentiment_dataset.csv')
        except:
            messagebox.showerror("Error", "Dataset not found.")

    def process(self):
        raw = self.entry.get()
        if not raw: return
        self.entry.delete(0, tk.END)
        self.ai_text.config(text="Searching your frequency...", fg="#6b7280")
        threading.Thread(target=self.generate_reply, args=(raw,)).start()

    def generate_reply(self, text):
        time.sleep(1) # Human-like reflection
        
        detected = "Relaxed"
        text_l = text.lower()
        
        # 1. Professional Sentiment Logic
        for mood, words in self.lexicon.items():
            if any(w in text_l for w in words):
                detected = mood
                break
        
        # 2. Get Song from Real Data
        subset = self.music_df[self.music_df['Sentiment_Label'] == detected]
        song = subset.sample(1).iloc[0] if not subset.empty else self.music_df.iloc[0]

        # 3. Gemini-Style Responses
        convos = {
            'Sad': f"I hear the weight in your words. It takes strength to express how low you're feeling. Sometimes, music is the only thing that understands. I've chosen '{song['Song_Name']}' by {song['Artist']} to sit with you in this moment.",
            'Happy': f"That energy is contagious! I'm vibrating on that same high frequency with you. Celebrate this joy—you've earned it. Here is '{song['Song_Name']}' by {song['Artist']} to keep this beautiful momentum going!",
            'Motivated': f"I can feel that fire! Your focus is razor-sharp right now. You're ready to conquer. I'm locking in '{song['Song_Name']}' by {song['Artist']} as the fuel for your grind. Go get it.",
            'Relaxed': f"Taking a moment to find your center is so important. I can feel you're looking for peace. Let's slow everything down. I recommend '{song['Song_Name']}' by {song['Artist']} to help you drift into that calm space."
        }

        self.root.after(0, lambda: self.refresh_ui(convos[detected], detected))

    def refresh_ui(self, msg, sentiment):
        self.ai_text.config(text=msg, fg='white')
        self.update_mood_image(sentiment)
        # Update text color based on mood
        self.ai_text.config(fg=self.colors[sentiment])

if __name__ == "__main__":
    root = tk.Tk()
    app = OmniEmotionPro(root)
    root.mainloop()

In [7]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from thefuzz import process, fuzz

class OmniEmotionMusicEngine:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL OMNI-CORE // SENTIMENT V7.0")
        self.root.geometry("1200x850")
        
        # PRO-TECH PALETTE
        self.colors = {
            'void': '#020205',
            'glass_panel': '#0c101c',
            'text_high': '#ffffff',
            'text_dim': '#6b7280',
            # Mood-Adaptive Accents
            'Happy': '#1DB954',    # Green
            'Sad': '#1e40af',      # Deep Blue
            'Relaxed': '#8b5cf6',  # Purple
            'Motivated': '#f97316' # Orange
        }
        
        # OMNI-EMOTION DICTIONARY (Training the Human Understanding)
        # We map dozens of nuances to your CSV's Sentiment_Label categories
        self.omni_feeling_lexicon = {
            'sad': ['lonely', 'miserable', 'gloomy', 'crying', 'heartbroken', 'hopeless', 'grieving', 'hurt', 'pain', 'burnout', 'nostalgic', 'homesick', 'down', 'empty'],
            'happy': ['excited', 'joyful', 'amazing', 'excellent', 'on top of the world', 'content', 'cheerful', 'wonderful', 'loving', 'grateful', 'proud', 'hype', 'pumped'],
            'relaxed': ['chill', 'stressed', 'tired', 'zen', 'calm', 'peaceful', 'quiet', 'meditating', 'sleepy', 'slow', 'overwhelmed', 'anxious', 'worried'],
            'motivated': ['ready', 'grind', 'gym', 'workout', 'strong', 'powerful', 'determined', 'focused', 'grinding', 'ambitious', 'energetic', 'confident', 'invincible']
        }

        self.music_df = None
        self.current_vibe = "CALIBRATING"
        self.setup_ui()
        self.load_engine()

    def setup_ui(self):
        self.root.configure(bg=self.colors['void'])

        # --- DYNAMIC BACKGROUND CANVAS ---
        # Where the visualizer lives
        self.bg_canvas = tk.Canvas(self.root, bg=self.colors['void'], highlightthickness=0)
        self.bg_canvas.place(x=0, y=0, relwidth=1, relheight=1)

        # --- INTERACTIVE CONTENT LAYER (Floating glass-morphism panel) ---
        self.display_panel = tk.Frame(self.root, bg=self.colors['glass_panel'], pady=40, padx=40)
        self.display_panel.place(relx=0.5, rely=0.45, anchor='center')

        # Sidebar HUD info
        self.hud_label = tk.Label(self.display_panel, text="// NEURAL RESPONSE FEED", font=('Consolas', 9), 
                                  fg=self.colors['text_dim'], bg=self.colors['glass_panel'])
        self.hud_label.pack(anchor='w', pady=(0, 20))

        self.resp_label = tk.Label(self.display_panel, 
                                   text="System initialized. Describe your feeling in detail. I am trained to listen...",
                                   font=('Segoe UI Light', 22), fg='white',
                                   bg=self.colors['glass_panel'], wraplength=700, justify="center")
        self.resp_label.pack(expand=True)
        
        # Simulated "Perfect Image" Area
        self.image_card = tk.Canvas(self.display_panel, width=200, height=200, 
                                     bg=self.colors['void'], highlightthickness=0)
        self.image_card.pack(pady=20)

        # --- COMMAND INPUT ---
        input_container = tk.Frame(self.root, bg=self.colors['void'], pady=30)
        input_container.pack(side=tk.BOTTOM, fill=tk.X)

        self.cmd_line = tk.Frame(input_container, bg=self.colors['text_dim'], padx=1, pady=1)
        self.cmd_line.pack(padx=150, fill=tk.X)

        self.user_input = tk.Entry(self.cmd_line, font=('Segoe UI', 18), 
                                   bg='#111827', fg='white', insertbackground='white',
                                   relief=tk.FLAT, justify='center')
        self.user_input.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=15, pady=12)
        self.user_input.bind("<Return>", lambda e: self.process_interaction())

        # Start background visualizer animation
        self.animate_visualizer()

    def load_engine(self):
        try:
            self.music_df = pd.read_csv('music_sentiment_dataset.csv')
            # Set initial status
            self.root.after(1000, lambda: self.hud_label.config(text="// CORE: ONLINE // SENTIMENT INDEXED // AWAITING FREQUENCY", fg=self.colors['Happy']))
        except:
            messagebox.showerror("Error", "Dataset 'music_sentiment_dataset.csv' not found.")

    def animate_visualizer(self):
        """Creates a professional colorful moving waveform based on the mood"""
        self.bg_canvas.delete("viz")
        w = self.root.winfo_width()
        h_max = 120
        # Color defaults to void until mood detected
        color = self.colors.get(self.current_vibe, '#111827')
        
        # Create bars that look like a professional EQ
        for i in range(0, w, 25):
            # The "energy" of the viz changes with mood
            vibe_energy = 80 if self.current_vibe in ['Happy', 'Motivated'] else 30
            h = random.randint(5, vibe_energy)
            
            self.bg_canvas.create_rectangle(i, 800-h, i+15, 800, fill=color, outline="", stipple='gray50', tags="viz")
        self.root.after(120, self.animate_visualizer)

    def generate_image_sim(self, sentiment):
        """Simulates 'Perfect Image' generation using geometric mood-shapes"""
        self.image_card.delete("art")
        color = self.colors.get(sentiment, self.colors['void'])
        
        if sentiment == 'Happy':
            # Pulsing sun-like circle
            self.image_card.create_oval(30, 30, 170, 170, fill=color, outline=self.colors['text_high'], width=2, tags="art")
            self.image_card.create_oval(70, 70, 130, 130, fill=self.colors['text_high'], outline="", tags="art")
        elif sentiment == 'Sad':
            # Misty glow / rainy look
            for i in range(10):
                r = random.randint(10, 80)
                self.image_card.create_oval(100-r, 100-r, 100+r, 100+r, outline=color, width=1, tags="art")
        elif sentiment == 'Motivated':
            # Sharp, upward-pointing triangles
            self.image_card.create_polygon(100, 20, 180, 180, 20, 180, fill=color, outline=self.colors['text_high'], width=2, tags="art")
        elif sentiment == 'Relaxed':
            # Flowing, horizontal waves
            for i in range(0, 200, 20):
                self.image_card.create_line(0, i, 200, i+10, fill=color, width=2, smooth=True, tags="art")

    def process_interaction(self):
        raw_text = self.user_input.get()
        if not raw_text: return
        self.user_input.delete(0, tk.END)
        
        self.resp_label.config(text="// PROCESSING OMNI-SENTIMENT...", fg=self.colors['text_dim'])
        # Threaded reply for responsive UI
        threading.Thread(target=self.generate_conversational_reply, args=(raw_text,)).start()

    def generate_conversational_reply(self, text):
        time.sleep(1) # Human-like reflection pause
        text_lower = text.lower()
        detected_sentiment = None
        
        # 1. Advanced OMNI-feeling scan
        for main_mood, synonyms in self.omni_feeling_lexicon.items():
            if any(word in text_lower for word in synonyms) or main_mood in text_lower:
                detected_sentiment = main_mood.capitalize()
                break
        
        if not detected_sentiment:
            detected_sentiment = "Relaxed" # Default calming fallback

        # 2. Gemini-like Empathetic Conversational Responses
        # Categorized by emotion
        conversations = {
            'Sad': [
                "I'm truly sorry you're navigating through that feeling. Please know I am here to listen. Sometimes, giving space to sadness through the right music can be a form of comfort. I found this track in your library that might resonate with your state right now...",
                "Thank you for trusting me with how you feel. It sounds incredibly heavy. When we feel overwhelmed by those gloomer emotions, music can act as a gentle companion. This song seems to match that frequency..."
            ],
            'Happy': [
                "That is wonderful! I am picking up on that incredible positive frequency you're radiating. Celebrate that joy—it's powerful. Let's amplify that feeling even more with this track I just found...",
                "Yes! I love hearing that. That kind of happiness deserves a soundtrack. I've scanned your library and this one feels like the perfect match to keep that energy flowing..."
            ],
            'Relaxed': [
                "It sounds like you are looking for a moment of peace, and that is completely understood. In a world that rarely stops moving, finding stillness is essential. Take a slow breath. This comforting track I found might help you drift away...",
                "I hear that need for calm. If you are feeling overwhelmed, it's okay to step back. Music can create a sanctuary for you to rest. I recommend this soothing song to help you find your center..."
            ],
            'Motivated': [
                "I can feel that fire! That level of focus and ambition is formidable. You are locked in, and I want to help you fuel that grind. This track I just prioritized is designed to match that intense Invincible energy...",
                "Let's go! That's the mindset of someone ready to achieve. You've got the determination, and now you just need the frequency. This song is perfect for pushing you through those final reps or that complex project..."
            ]
        }

        # Select a real song from the data
        subset = self.music_df[self.music_df['Sentiment_Label'] == detected_sentiment]
        if subset.empty:
            subset = self.music_df # Final fallback

        # 3. Apply the results
        self.current_vibe = detected_sentiment
        match = subset.sample(1).iloc[0]
        
        # Prepare the Gemini-style response block
        response = f"{random.choice(conversations[detected_sentiment])}\n\n"
        response += f"🎵 **'{match['Song_Name']}'**\n👤 {match['Artist']}\n—\nA {match['Mood'].lower()} {match['Genre']} composition.\n\n"
        response += f"STATUS: [FREQUENCIES ALIGNED // VISUALIZER CALIBRATED // ART GENERATED]"
        
        # Update UI from the main thread
        self.root.after(0, lambda: self.update_dashboard(response, detected_sentiment))

    def update_dashboard(self, response, sentiment):
        # Update the main conversation
        self.resp_label.config(text=response, fg='white')
        
        # Update Sidebar HUD status
        color = self.colors.get(sentiment, self.colors['Happy'])
        self.hud_label.config(text=f"// FREQUENCY: {sentiment.upper()} // ARTIST: CONFIRMED // VIBE: ESTABLISHED", fg=color)
        
        # Create the simulated "Perfect Image"
        self.generate_image_sim(sentiment)

if __name__ == "__main__":
    root = tk.Tk()
    app = OmniEmotionMusicEngine(root)
    root.mainloop()

In [8]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from thefuzz import process, fuzz

class OmniEmotionBookEngine:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL LITERARY CORE // BIBLIOPHILE V9.0")
        self.root.geometry("1200x850")
        
        # PRO-TECH PALETTE (Dark Academia / Cyberpunk Hybrid)
        self.colors = {
            'void': '#0a0a0c',          # Ink Black
            'glass_panel': '#16161d',   # Deep Charcoal
            'text_high': '#f1f1f1',     # Paper White
            'text_dim': '#7a7a8a',      # Dust Grey
            # Mood-Adaptive Accents
            'Adventure': '#f97316',     # Orange (Excited)
            'Melancholy': '#3b82f6',    # Blue (Sad)
            'Curiosity': '#8b5cf6',     # Purple (Relaxed)
            'Intensity': '#ef4444'      # Red (Motivated)
        }
        
        # OMNI-EMOTION TO GENRE MAPPING
        self.emotion_to_genre = {
            'sad': ['Classics', 'Historical Fiction', 'Poetry', 'Drama'],
            'happy': ['Humor', 'Romance', 'Contemporary', 'Middle Grade'],
            'relaxed': ['Fantasy', 'Fairy Tales', 'Nature', 'Art'],
            'motivated': ['Nonfiction', 'Business', 'Self Help', 'Thriller', 'Science Fiction', 'Action']
        }

        self.books_df = None
        self.current_vibe = "Adventure"
        self.setup_ui()
        self.load_engine()

    def setup_ui(self):
        self.root.configure(bg=self.colors['void'])

        # --- DYNAMIC BACKGROUND CANVAS ---
        self.bg_canvas = tk.Canvas(self.root, bg=self.colors['void'], highlightthickness=0)
        self.bg_canvas.place(x=0, y=0, relwidth=1, relheight=1)

        # --- INTERACTIVE CONTENT LAYER ---
        self.display_panel = tk.Frame(self.root, bg=self.colors['glass_panel'], pady=40, padx=40)
        self.display_panel.place(relx=0.5, rely=0.45, anchor='center')

        self.hud_label = tk.Label(self.display_panel, text="// LITERARY FEED INITIALIZING...", font=('Consolas', 9), 
                                  fg=self.colors['text_dim'], bg=self.colors['glass_panel'])
        self.hud_label.pack(anchor='w', pady=(0, 20))

        self.resp_label = tk.Label(self.display_panel, 
                                   text="Tell me of your journey or your burdens. I shall find the pages you need...",
                                   font=('Times New Roman', 22, 'italic'), fg='white',
                                   bg=self.colors['glass_panel'], wraplength=750, justify="center")
        self.resp_label.pack(expand=True)
        
        # Abstract "Book Cover" Art Area
        self.book_art = tk.Canvas(self.display_panel, width=180, height=260, 
                                     bg='#000000', highlightthickness=1, highlightbackground=self.colors['text_dim'])
        self.book_art.pack(pady=20)

        # --- COMMAND INPUT ---
        input_container = tk.Frame(self.root, bg=self.colors['void'], pady=30)
        input_container.pack(side=tk.BOTTOM, fill=tk.X)

        self.cmd_line = tk.Frame(input_container, bg=self.colors['text_dim'], padx=1, pady=1)
        self.cmd_line.pack(padx=150, fill=tk.X)

        self.user_input = tk.Entry(self.cmd_line, font=('Segoe UI', 18), 
                                   bg='#111827', fg='white', insertbackground='white',
                                   relief=tk.FLAT, justify='center')
        self.user_input.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=15, pady=12)
        self.user_input.bind("<Return>", lambda e: self.process_interaction())

        self.animate_ambience()

    def animate_ambience(self):
        """Creates a professional 'floating parchment' background effect"""
        self.bg_canvas.delete("ink")
        w = self.root.winfo_width()
        color = self.colors.get(self.current_vibe, '#1e1e2e')
        
        # Draw floating 'particles' of ink/parchment
        for _ in range(12):
            x = random.randint(0, w)
            y = random.randint(0, 800)
            size = random.randint(2, 6)
            self.bg_canvas.create_oval(x, y, x+size, y+size, fill=color, outline="", stipple='gray25', tags="ink")
        self.root.after(150, self.animate_ambience)

    def draw_book_cover(self, sentiment):
        """Generates a geometric 'Book Cover' based on sentiment"""
        self.book_art.delete("cover")
        color = self.colors.get(sentiment, '#ffffff')
        
        # Base book spine
        self.book_art.create_rectangle(10, 10, 170, 250, outline=color, width=2, tags="cover")
        
        if sentiment == 'Melancholy':
            for i in range(20, 240, 10):
                self.book_art.create_line(30, i, 150, i, fill=color, tags="cover")
        elif sentiment == 'Adventure':
            self.book_art.create_polygon(90, 50, 150, 200, 30, 200, fill=color, tags="cover")
        elif sentiment == 'Intensity':
            for _ in range(20):
                x, y = random.randint(20, 160), random.randint(20, 240)
                self.book_art.create_oval(x, y, x+5, y+5, fill=color, tags="cover")

    def load_engine(self):
        try:
            self.books_df = pd.read_csv('books_1.Best_Books_Ever.csv')
            self.hud_label.config(text="// ARCHIVES UNLOCKED // 50,000+ VOLUMES INDEXED // AWAITING INTENT", fg="#1DB954")
        except:
            messagebox.showerror("Error", "Dataset 'books_1.Best_Books_Ever.csv' not found.")

    def process_interaction(self):
        raw_text = self.user_input.get()
        if not raw_text: return
        self.user_input.delete(0, tk.END)
        self.resp_label.config(text="// CONSULTING THE ARCHIVES...", fg=self.colors['text_dim'])
        threading.Thread(target=self.generate_literary_reply, args=(raw_text,)).start()

    def generate_literary_reply(self, text):
        time.sleep(1.5)
        text_lower = text.lower()
        
        # Sentiment mapping to genre categories
        sentiment = 'Curiosity'
        target_genres = self.emotion_to_genre['relaxed']
        
        if any(word in text_lower for word in ['sad', 'lonely', 'cry', 'dark']):
            sentiment = 'Melancholy'
            target_genres = self.emotion_to_genre['sad']
        elif any(word in text_lower for word in ['happy', 'fun', 'love', 'laugh']):
            sentiment = 'Adventure'
            target_genres = self.emotion_to_genre['happy']
        elif any(word in text_lower for word in ['power', 'gym', 'work', 'fast']):
            sentiment = 'Intensity'
            target_genres = self.emotion_to_genre['motivated']

        # Filter books based on genre list
        # We look for rows where the 'genres' string contains one of our target keywords
        mask = self.books_df['genres'].str.contains('|'.join(target_genres), na=False, case=False)
        subset = self.books_df[mask]
        
        if subset.empty:
            subset = self.books_df

        self.current_vibe = sentiment
        book = subset.sample(1).iloc[0]
        
        # Construct Gemini-style literary response
        desc_snippet = str(book['description'])[:200] + "..."
        response = f"I perceive your state as one of {sentiment}. In times like these, the soul often seeks sanctuary in specific narratives.\n\n"
        response += f"📖 **{book['title']}**\n✍️ {book['author']}\n⭐ Rating: {book['rating']}/5\n\n"
        response += f"\"{desc_snippet}\""
        
        self.root.after(0, lambda: self.update_dashboard(response, sentiment))

    def update_dashboard(self, response, sentiment):
        self.resp_label.config(text=response, fg='white', font=('Times New Roman', 14))
        color = self.colors.get(sentiment, '#ffffff')
        self.hud_label.config(text=f"// RECOMMENDATION: {sentiment.upper()} // GENRE MATCHED // ARCHIVE SELECTED", fg=color)
        self.draw_book_cover(sentiment)

if __name__ == "__main__":
    root = tk.Tk()
    app = OmniEmotionBookEngine(root)
    root.mainloop()

In [5]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import threading
import time
import random
from datetime import datetime
from thefuzz import process, fuzz

class UltimateLiteraryOracle:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL OMNI-ORACLE V11.0 // TOTAL IMMERSION")
        self.root.geometry("1200x900")
        
        # OMNI-GENRE COLOR PALETTE
        self.colors = {
            'void': '#020205',
            'glass': '#0c101c',
            'Default': '#6b7280',
            'Horror': '#ff0000',       # Blood Red
            'Fantasy': '#ffd700',      # Magic Gold
            'Sci-Fi': '#00f2ff',       # Cyber Cyan
            'Romance': '#ff69b4',      # Rose Pink
            'Classics': '#cd853f',     # Parchment Brown
            'Thriller': '#ffffff',     # Stark White
            'Action': '#ff4500'        # Explosion Orange
        }
        
        # HUMAN EMOTION TO MULTI-GENRE MAPPING
        self.omni_map = {
            'Fear': ['Horror', 'Paranormal', 'Gothic', 'Ghosts'],
            'Wonder': ['Fantasy', 'Magic', 'Mythology', 'Fairy Tales'],
            'Future': ['Science Fiction', 'Dystopia', 'Cyberpunk', 'Space'],
            'Love': ['Romance', 'Contemporary Romance', 'Young Adult'],
            'Nostalgia': ['Classics', 'Historical Fiction', 'Literature'],
            'Adrenaline': ['Action', 'Thriller', 'Mystery', 'Crime', 'Suspense']
        }

        self.books_df = None
        self.current_vibe = "Default"
        self.setup_ui()
        self.load_engine()

    def setup_ui(self):
        self.root.configure(bg=self.colors['void'])

        # --- DYNAMIC BACKGROUND CANVAS ---
        self.bg_canvas = tk.Canvas(self.root, bg=self.colors['void'], highlightthickness=0)
        self.bg_canvas.place(x=0, y=0, relwidth=1, relheight=1)

        # --- FLOATING HUD PANEL ---
        self.panel = tk.Frame(self.root, bg=self.colors['glass'], pady=40, padx=40)
        self.panel.place(relx=0.5, rely=0.45, anchor='center')

        self.hud_status = tk.Label(self.panel, text="// SYSTEM READY: AWAITING HUMAN INTENT", 
                                   font=('Consolas', 9), fg=self.colors['Default'], bg=self.colors['glass'])
        self.hud_status.pack(anchor='w', pady=(0, 20))

        self.resp_label = tk.Label(self.panel, 
                                   text="Welcome to the Archive. What story does your soul seek?",
                                   font=('Segoe UI Light', 20), fg='white',
                                   bg=self.colors['glass'], wraplength=750, justify="center")
        self.resp_label.pack(expand=True)
        
        # Central Visual Generator
        self.art_card = tk.Canvas(self.panel, width=250, height=200, 
                                     bg=self.colors['void'], highlightthickness=0)
        self.art_card.pack(pady=20)

        # --- INPUT INTERFACE ---
        input_box = tk.Frame(self.root, bg=self.colors['void'], pady=30)
        input_box.pack(side=tk.BOTTOM, fill=tk.X)

        self.entry_frame = tk.Frame(input_box, bg=self.colors['Default'], padx=1, pady=1)
        self.entry_frame.pack(padx=150, fill=tk.X)

        self.user_input = tk.Entry(self.entry_frame, font=('Segoe UI', 18), 
                                   bg='#0a0a14', fg='white', insertbackground='white',
                                   relief=tk.FLAT, justify='center')
        self.user_input.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=15, pady=12)
        self.user_input.bind("<Return>", lambda e: self.trigger_search())

        self.animate_world()

    def animate_world(self):
        """Creates a professional animated background that changes with the genre"""
        self.bg_canvas.delete("env")
        w = self.root.winfo_width()
        color = self.colors.get(self.current_vibe, self.colors['Default'])
        
        # Dynamic Particle System
        for _ in range(15):
            x, y = random.randint(0, w), random.randint(0, 900)
            size = random.randint(1, 4)
            
            if self.current_vibe == 'Horror':
                # Jagged Static
                self.bg_canvas.create_line(x, y, x, y+20, fill=color, tags="env")
            elif self.current_vibe == 'Fantasy':
                # Glowing Sparkles
                self.bg_canvas.create_oval(x, y, x+size, y+size, fill=color, outline="white", tags="env")
            elif self.current_vibe == 'Sci-Fi':
                # Vertical Matrix Drips
                self.bg_canvas.create_line(x, y, x, y+40, fill=color, stipple='gray50', tags="env")
            else:
                # Floating Dust
                self.bg_canvas.create_oval(x, y, x+size, y+size, fill=color, stipple='gray25', tags="env")
        
        self.root.after(100, self.animate_world)

    def draw_art(self, vibe):
        self.art_card.delete("art")
        color = self.colors.get(vibe, 'white')
        
        if vibe == 'Horror':
            for _ in range(40):
                x = random.randint(0, 250)
                self.art_card.create_line(x, 0, x, random.randint(20, 200), fill=color, tags="art")
        elif vibe == 'Fantasy':
            self.art_card.create_oval(75, 50, 175, 150, outline=color, width=3, tags="art")
            self.art_card.create_text(125, 100, text="✨", fill=color, font=("Arial", 40), tags="art")
        elif vibe == 'Romance':
            self.art_card.create_text(125, 100, text="❤", fill=color, font=("Arial", 60), tags="art")
        elif vibe == 'Sci-Fi':
            for i in range(10, 100, 10):
                self.art_card.create_rectangle(125-i, 100-i, 125+i, 100+i, outline=color, tags="art")

    def load_engine(self):
        try:
            self.books_df = pd.read_csv('books_1.Best_Books_Ever.csv')
        except:
            messagebox.showerror("Error", "Dataset not found.")

    def trigger_search(self):
        query = self.user_input.get()
        if not query: return
        self.user_input.delete(0, tk.END)
        self.resp_label.config(text="// SCANNING THE OMNI-ARCHIVE...", fg=self.colors['Default'])
        threading.Thread(target=self.neural_logic, args=(query,)).start()

    def neural_logic(self, text):
        time.sleep(1.2)
        text_l = text.lower()
        
        # 1. Advanced Intent Detection
        detected_vibe = "Default"
        for vibe, keywords in self.omni_map.items():
            if any(k.lower() in text_l for k in keywords) or vibe.lower() in text_l:
                # Map internal Vibe to UI Color keys
                vibe_map = {'Fear': 'Horror', 'Wonder': 'Fantasy', 'Future': 'Sci-Fi', 'Love': 'Romance', 'Nostalgia': 'Classics', 'Adrenaline': 'Action'}
                detected_vibe = vibe_map[vibe]
                break

        # 2. Hybrid Search: CSV + World Knowledge
        # Search the CSV for genres related to the vibe
        subset = self.books_df[self.books_df['genres'].str.contains(detected_vibe, na=False, case=False)]
        
        if subset.empty:
            # Emergency World-Knowledge Bestsellers
            world_library = {
                'Horror': {"title": "Dracula", "author": "Bram Stoker", "desc": "The foundational vampire mythos..."},
                'Fantasy': {"title": "The Name of the Wind", "author": "Patrick Rothfuss", "desc": "A young man grows into the most notorious wizard..."},
                'Sci-Fi': {"title": "Dune", "author": "Frank Herbert", "desc": "A messianic tale on a desert planet..."},
                'Romance': {"title": "The Notebook", "author": "Nicholas Sparks", "desc": "An enduring tale of love across time..."}
            }
            book = world_library.get(detected_vibe, {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "desc": "A classic of the Jazz Age."})
        else:
            row = subset.sample(1).iloc[0]
            book = {"title": row['title'], "author": row['author'], "desc": str(row['description'])[:200]}

        # 3. Gemini-Style Response Construction
        intros = {
            'Horror': "I sense a curiosity for the darkness. Some stories are warnings; others are mirrors. Be careful—this one is both.",
            'Fantasy': "You wish to leave this world behind. Excellent. I have found a gateway built of words and wonder.",
            'Sci-Fi': "The future is not a destination, but a possibility. I have calculated the perfect narrative for your mechanical heart.",
            'Romance': "Ah, the complexities of the human heart. It is a fragile thing, yet it drives every great story. Listen to this..."
        }
        
        response = f"{intros.get(detected_vibe, 'I have found a volume that speaks to your intent.')}\n\n"
        response += f"📖 **{book['title']}**\n👤 {book['author']}\n\n"
        response += f"\"{book['desc']}...\""
        
        self.current_vibe = detected_vibe
        self.root.after(0, lambda: self.update_ui(response, detected_vibe))

    def update_ui(self, msg, vibe):
        self.resp_label.config(text=msg, fg='white')
        color = self.colors.get(vibe, 'white')
        self.hud_status.config(text=f"// FREQUENCY: {vibe.upper()} // GENRE: STABLE // ARCHIVE: ACCESSED", fg=color)
        self.draw_art(vibe)

if __name__ == "__main__":
    root = tk.Tk()
    app = UltimateLiteraryOracle(root)
    root.mainloop()

In [6]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import pandas as pd
import re
import threading
import time

# --- AI CORE LIBRARIES ---
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import linear_kernel
except ImportError:
    import os
    os.system('pip install scikit-learn pandas')
    messagebox.showwarning("System Update", "Installing AI modules... Please restart the application in 1 minute.")

class LibreBotPro:
    def __init__(self, root):
        self.root = root
        self.root.title("LibreBot Pro - Neural Recommendation Engine")
        self.root.geometry("1100x850")
        self.root.configure(bg='#0f172a')
        
        # Engine Variables
        self.books_df = None
        self.tfidf_matrix = None
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
        self.is_typing = False
        
        self.setup_ui()
        # Start AI Engine in Background
        threading.Thread(target=self.initialize_ai_engine, daemon=True).start()

    def initialize_ai_engine(self):
        """Builds the TF-IDF Semantic Map for the dataset"""
        try:
            filename = 'books_1.Best_Books_Ever (1).csv'
            # 1. Load Data
            df = pd.read_csv(filename)
            
            # 2. Professional Data Cleaning
            df['description'] = df['description'].fillna('').astype(str)
            df['genres'] = df['genres'].fillna('').astype(str)
            df['author'] = df['author'].fillna('Unknown').astype(str)
            df['title'] = df['title'].fillna('Untitled').astype(str)
            
            # 3. Create 'Knowledge Base' for each book
            # We combine these fields so the AI understands relationships between them
            content_series = df['title'] + " " + df['author'] + " " + df['genres'] + " " + df['description']
            
            # 4. Generate Vector Matrix (The "Brain")
            self.tfidf_matrix = self.vectorizer.fit_transform(content_series)
            self.books_df = df
            
            self.update_status(f"● AI ENGINE ONLINE | {len(df):,} VOLUMES INDEXED")
            self.root.after(500, self.send_welcome_msg)
        except Exception as e:
            self.update_status(f"❌ ENGINE ERROR: {str(e)}")
            print(f"Detailed Error: {e}")

    def get_ai_recommendations(self, query):
        """Calculates Cosine Similarity between user query and book database"""
        # Transform user input into the same mathematical space as the books
        query_vec = self.vectorizer.transform([query])
        
        # 
        # Calculate how similar the query is to every single book (0.0 to 1.0)
        cosine_sim = linear_kernel(query_vec, self.tfidf_matrix).flatten()
        
        # Get indices of the top 10 most similar books
        top_indices = cosine_sim.argsort()[:-11:-1]
        
        # Return those books, filtered for quality
        results = self.books_df.iloc[top_indices].copy()
        return results.sort_values(by='rating', ascending=False).head(5)

    def setup_ui(self):
        # Header / Branding
        header = tk.Frame(self.root, bg='#1e293b', height=100)
        header.pack(fill=tk.X)
        header.pack_propagate(False)
        
        tk.Label(header, text="LIBREBOT PRO", font=('Inter', 24, 'bold'), fg='#38bdf8', bg='#1e293b').pack(pady=(20, 0))
        tk.Label(header, text="SEMANTIC SEARCH & NEURAL RECOMMENDATION", font=('Inter', 8, 'bold'), fg='#94a3b8', bg='#1e293b').pack()

        # Chat Interface
        self.display = scrolledtext.ScrolledText(self.root, wrap=tk.WORD, font=('Inter', 11), bg='#0f172a', 
                                               fg='#f1f5f9', relief=tk.FLAT, padx=30, pady=20)
        self.display.pack(fill=tk.BOTH, expand=True)
        self.display.config(state=tk.DISABLED)

        # Input Zone
        ctrl_frame = tk.Frame(self.root, bg='#1e293b', pady=20, padx=30)
        ctrl_frame.pack(fill=tk.X)

        self.input_entry = tk.Entry(ctrl_frame, font=('Inter', 12), bg='#334155', fg='white', 
                                  insertbackground='white', relief=tk.FLAT)
        self.input_entry.pack(fill=tk.X, side=tk.LEFT, expand=True, ipady=10, padx=(0, 15))
        self.input_entry.bind('<Return>', lambda e: self.handle_input())

        btn = tk.Button(ctrl_frame, text="RUN ANALYSIS", command=self.handle_input, bg='#38bdf8', 
                       fg='#0f172a', font=('Inter', 9, 'bold'), relief=tk.FLAT, padx=20)
        btn.pack(side=tk.RIGHT, ipady=8)

        self.status_lbl = tk.Label(self.root, text="● WARMING UP AI MODULES...", bg='#020617', fg='#64748b', font=('Inter', 8), anchor=tk.W, padx=20)
        self.status_lbl.pack(fill=tk.X)

    def handle_input(self):
        query = self.input_entry.get().strip()
        if query:
            self.input_entry.delete(0, tk.END)
            self.add_msg("USER", query)
            threading.Thread(target=self.generate_response, args=(query,), daemon=True).start()

    def generate_response(self, query):
        self.show_typing()
        time.sleep(0.8)
        
        if self.books_df is None:
            self.add_msg("SYSTEM", "Engine not ready. Please wait for indexing to complete.")
            return

        try:
            recs = self.get_ai_recommendations(query)
            response = "### NEURAL MATCHES FOUND\nBased on your prompt, these titles share the highest semantic similarity:\n\n"
            for i, (_, row) in enumerate(recs.iterrows(), 1):
                response += f"{i}. **{row['title']}**\n   - By {row['author']}\n   - Rating: ⭐ {row['rating']}\n   - Genre: {str(row['genres'])[:80]}...\n\n"
            
            self.root.after(0, lambda: self.add_msg("LIBREBOT", response))
        except Exception as e:
            self.root.after(0, lambda: self.add_msg("LIBREBOT", f"Error during analysis: {e}"))

    def add_msg(self, sender, text):
        self.display.config(state=tk.NORMAL)
        if self.is_typing:
            self.display.delete("end-2l", tk.END)
            self.display.insert(tk.END, "\n")
            self.is_typing = False
        
        color = "#38bdf8" if sender == "LIBREBOT" else "#10b981"
        self.display.insert(tk.END, f"[{sender}]\n", sender)
        
        # Bold formatting
        parts = re.split(r'(\*\*.*?\*\*)', text)
        for part in parts:
            if part.startswith('**'): self.display.insert(tk.END, part.replace('**',''), "bold")
            else: self.display.insert(tk.END, part)
            
        self.display.insert(tk.END, "\n\n")
        self.display.tag_config("LIBREBOT", foreground="#38bdf8", font=('Inter', 10, 'bold'))
        self.display.tag_config("USER", foreground="#10b981", font=('Inter', 10, 'bold'))
        self.display.tag_config("bold", font=('Inter', 11, 'bold'))
        self.display.config(state=tk.DISABLED)
        self.display.see(tk.END)

    def show_typing(self):
        self.is_typing = True
        self.display.config(state=tk.NORMAL)
        self.display.insert(tk.END, "[LIBREBOT]\n", "LIBREBOT")
        self.display.insert(tk.END, "Analyzing textual vectors...\n", "typing")
        self.display.tag_config("typing", foreground="#64748b", font=('Inter', 10, 'italic'))
        self.display.config(state=tk.DISABLED)

    def update_status(self, msg):
        self.status_lbl.config(text=msg)

    def send_welcome_msg(self):
        self.add_msg("LIBREBOT", "System Online. I am **LibreBot Pro**. Give me a description, a book you like, or a theme, and I will find the most mathematically similar matches.")

if __name__ == "__main__":
    root = tk.Tk()
    app = LibreBotPro(root)
    root.mainloop()

Detailed Error: [Errno 2] No such file or directory: 'books_1.Best_Books_Ever (1).csv'
